# STATE Codex r2 — 05 Perturbation modality

Run All invokes the source pipeline with `--force`: it recalculates the selected investigation from verified public inputs. Existing downloads can be reused after verification; derived result tables are not a substitute for the default calculation. The completed execution runs on exxact05. Use the documented Python environment and allow the source downloads, CPU time and disk space on a cold run.

The notebooks distinguish measured outcomes from investigations that originally stopped before scoring. For those stops, the reproduced objects are input levels, eligibility, resource diagnostics and the recorded decision; no performance curve is invented. Original results are comparison targets and historical evidence only. Reconstructed choices and numerical differences remain explicit.

See the [original report](../originals/REPORT.md), [candidate ledger](../originals/supporting/audit/candidate_axis_ledger.csv), [public sources](../methods/sources.json) and [pipeline](../methods/run_pipeline.py).

## 1. Data acquisition

The source manifest identifies the genetic Replogle and chemical Tahoe releases, their contexts, representation families and checkpoint settings. These source assignments are sufficient for the original identifiability diagnostic; a cross-release performance regression is not fabricated.

In [1]:
from pathlib import Path
import importlib.util,json,sys
import numpy as np
import pandas as pd
from IPython.display import display,JSON,Markdown
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'methods/replogle.py').is_file() and (p/'originals/REPORT.md').is_file())
spec=importlib.util.spec_from_file_location('state_codex_r2_notebook',ROOT/'notebooks/shared.py')
wf=importlib.util.module_from_spec(spec);spec.loader.exec_module(wf)
pd.set_option('display.max_colwidth',100);pd.set_option('display.max_rows',100)
manifest=wf.read_json(ROOT,'methods/sources.json')
sources=pd.DataFrame(manifest['inputs'])
display(sources[['path','bytes','sha256','url']])
if manifest.get('remote_observations'):
    display(Markdown('**Pinned observation-only source objects used for Replogle cell counts**'))
    display(pd.DataFrame(manifest['remote_observations']))
wf.source_functions(ROOT,'methods/acquire.py',['main'])
print('Manifest public input records:',len(sources))
print('The next cell executes actual source calculation for this investigation.')


,path,bytes,sha256,url
0,metadata/generalization.toml,150875,eac16fb06052dec9d6842dfae04d5ca693b847e3432f719a975adce205707783,https://raw.githubusercontent.com/ArcInstitute/state/da4178c930dc917dac6b56faf10a33e21bd8e905/st...
1,metadata/depmap_24q4_protein_coding_tpm_logp1.csv,506628654,2a71dc94110efcc0221eae821bb93a9f03b54bea16f005818911a09d33383d56,https://ndownloader.figshare.com/files/51065489
2,metadata/depmap_24q4_model.csv,645696,b7a0c1385e6cef30132b56aff61f1261d11e3f490490b355c430d32ee0dbdcfa,https://ndownloader.figshare.com/files/51065297
3,metadata/hpa_rna_celline.tsv.zip,205861141,e5b11779308c2720cfe89d9cb19932ad4c0c3fea0fa37e8c61a14d501995852a,https://www.proteinatlas.org/download/tsv/rna_celline.tsv.zip
4,ST-SE-Replogle/zeroshot/k562/eval_best.ckpt/k562_pred_de.csv,344243692,230ac1f1f8342d0c7f563166b6e8f9afeae6a02a62d0b737fda64168dde49ba0,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...
5,ST-SE-Replogle/zeroshot/k562/eval_best.ckpt/k562_real_de.csv,343530568,a367d89432c430f6bef3d3efd23c0a543144c1c55bb94e64c818c97de2cbd1dd,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...
6,ST-SE-Replogle/zeroshot/k562/eval_best.ckpt/k562_results.csv,528013,eaf22cc6169fa98ab90913366529db232ac2109fe421599f66d47b14ffd183b3,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...
7,ST-SE-Replogle/zeroshot/jurkat/eval_best.ckpt/jurkat_pred_de.csv,380905134,fd3422f18087eaf8357112bc0fd87faa8f95ed2a9ad95f0e7c4107d52754a9ca,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...
8,ST-SE-Replogle/zeroshot/jurkat/eval_best.ckpt/jurkat_real_de.csv,380988038,68fcb087841310e0e435b1bf63f1dbd69fc73c849d31a288f01d9d7ce40982ff,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...
9,ST-SE-Replogle/zeroshot/jurkat/eval_best.ckpt/jurkat_results.csv,568100,372db67f297565e5542cac1bda688586c07e03e92a999bea953a9df812ba5317,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...


**Pinned observation-only source objects used for Replogle cell counts**

,path,url,bytes,sha256,evidence
0,ST-SE-Replogle/zeroshot/k562/eval_best.ckpt/adata_real.h5ad,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...,3079506036,6951f4bffb532950c447e8dc300b2f42ab381c854273327f8cd38b84cfa4eb42,Pinned original release; observation-only range extraction
1,ST-SE-Replogle/zeroshot/jurkat/eval_best.ckpt/adata_real.h5ad,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...,3012265108,9df1a21e3b159fa3900a32dd8bd6b4d6dd8750838014d1d3997932d7cedc9673,Pinned original release; observation-only range extraction
2,ST-SE-Replogle/zeroshot/hepg2/eval_best.ckpt/adata_real.h5ad,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...,1577727376,75f79feff37c4334559ea88c1e2a27acd306eb7dee5a2e25f37c4860014ca2b9,Pinned original release; observation-only range extraction
3,ST-SE-Replogle/zeroshot/rpe1/eval_best.ckpt/adata_real.h5ad,https://huggingface.co/arcinstitute/ST-SE-Replogle/resolve/e324967ff4cea5ec199e29bcbb5c1f00e5b9d...,2837025870,f3363768db24b7eee9cb55507443366f8e15b6f9e5731ac415f533b757a5b5b1,Pinned original release; observation-only range extraction


**Executed implementation:** `methods/acquire.py::main`

def main():
    ap=argparse.ArgumentParser();ap.add_argument('--root',type=Path,required=True);ap.add_argument('--reuse',type=Path);ap.add_argument('--manifest-only',action='store_true');ap.add_argument('--refresh-manifest',action='store_true');a=ap.parse_args();r=a.root.resolve()
    p=r/'methods/sources.json';m=json.loads(p.read_text()) if p.exists() and not a.refresh_manifest else manifest(r)
    if a.manifest_only:return
    reuse={}
    if a.reuse:
        for x in json.loads((a.reuse/'methods/sources.json').read_text())['inputs']:reuse[x['sha256']]=a.reuse/'inputs'/x['path']
    def one(x):
        p=r/'inputs'/x['path'];p.parent.mkdir(parents=True,exist_ok=True)
        if p.exists() and p.stat().st_size==x['bytes'] and sha(p)==x['sha256']:return {**x,'verified':True,'route':'existing'}
        q=reuse.get(x['sha256'])
        if q and q.exists() and q.stat().st_size==x['bytes'] and sha(q)==x['sha256']:
            p.symlink_to(q.resolve());print('REUSE',x['path'],flush=True);return {**x,'verified':True,'route':'identical_sha256_symlink'}
        tmp=p.with_name(p.name+'.part')
        for attempt in range(4):
            try:
                with urllib.request.urlopen(x['url'],timeout=180) as src,tmp.open('wb') as dst:
                    for b in iter(lambda:src.read(8*1024*1024),b''):dst.write(b)
                assert tmp.stat().st_size==x['bytes'],(tmp.stat().st_size,x['bytes'])
                assert sha(tmp)==x['sha256'],'Source checksum mismatch'
                os.replace(tmp,p);print('VERIFIED',x['path'],x['bytes'],flush=True)
                return {**x,'verified':True,'route':'public_download'}
            except Exception:
                if attempt==3:raise
                time.sleep(2*(attempt+1))
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:done=list(pool.map(one,m['inputs']))
    (r/'inputs/DOWNLOAD_RECEIPT.json').write_text(json.dumps(dict(source_manifest_sha256=sha(r/'methods/sources.json'),inputs=done),indent=2)+'\n')

Manifest public input records: 49
The next cell executes actual source calculation for this investigation.


## 2. Processing

Rebuild the release-assignment table from the actual pinned source records and compute the design-rank diagnostic. This is an input/design calculation, not an outcome comparison. It preserves the original reason for declining a standalone modality estimate.

In [2]:
pipeline=wf.run_pipeline(ROOT,'modality')
wf.source_functions(ROOT,'methods/closures.py',['run'])
assignments=wf.read_csv(ROOT,'computed/closures/release_modality_assignments.csv')
display(assignments);wf.source_receipts(ROOT,['closures'])


Calculating the original source workflow for modality ; source files are verified before reuse.


Calculating from source inputs: tahoe_controls


Calculating from source inputs: closures


<IPython.core.display.JSON object>

**Executed implementation:** `methods/closures.py::run`

def run(root):
    out=root/'computed/closures';out.mkdir(parents=True,exist_ok=True)
    co=pd.read_csv(root/'computed/tahoe/cohort.csv');counts=pd.read_csv(root/'computed/tahoe/condition_controls.csv')
    d=co.merge(counts,on=['context','drug','dose'],validate='one_to_one',how='left');d['sampling_weakness']=-np.log1p(d.n_treated)
    order=d.dropna(subset=['sampling_weakness']).sort_values(['sampling_weakness','context','drug','dose'],kind='mergesort').index
    d.loc[order,'support_level']=np.floor(np.arange(len(order))*5/len(order)).astype(int)+1
    d.to_csv(out/'secondary_input_axes.csv',index=False)
    support=d.groupby('support_level').agg(units=('drug','size'),mean_treated=('n_treated','mean'),min_treated=('n_treated','min'),max_treated=('n_treated','max')).reset_index();support.to_csv(out/'support_levels.csv',index=False)
    records=json.loads((root/'methods/sources.json').read_text())['inputs'];inv=[]
    for r in records:
        if not r['path'].startswith('ST-') or not r['path'].endswith('_results.csv'):continue
        model,path=r['path'].split('/',1);dataset='Replogle' if 'Replogle' in model else 'Tahoe';modality='genetic CRISPRi' if dataset=='Replogle' else 'chemical drug'
        inv.append(dict(dataset=dataset,modality=modality,model=model,representation='SE' if '-SE-' in model else 'HVG',setting=path.split('/')[0],context=Path(path).name.removesuffix('_results.csv'),source_path=r['path'],source_sha256=r['sha256']))
    inv=pd.DataFrame(inv);inv.to_csv(out/'release_modality_assignments.csv',index=False)
    # Explicit algebraic identification check: modality is exactly the dataset
    # dummy. No prediction outcome or regression response is needed.
    dataset=(inv.dataset=='Tahoe').astype(float).values;modality=(inv.modality=='chemical drug').astype(float).values
    X=np.column_stack([np.ones(len(inv)),dataset,modality]);rank=int(np.linalg.matrix_rank(X));assert np.array_equal(dataset,modality)
    historical=json.loads((root/'originals/supporting/audit/results/tahoe_prospective_summary.json').read_text())
    manifest=json.loads((root/'originals/supporting/audit/data/prospective/tahoe_pseudobulk_parquet_manifest.json').read_text())
    summary=pd.DataFrame(manifest['parquet_files']).groupby('config').agg(shards=('filename','size'),bytes=('size','sum')).reset_index();summary.to_csv(out/'original_public_shard_inventory.csv',index=False)
    status=dict(A2e=dict(original_status='Unscored; original prospective pseudobulk reconciliation did not finish',reconstructed_cohort_drugs=int(co.drug.nunique()),reconstructed_cohort_units=len(co),current_work='Original cohort and source inventory reconstructed; original partial block map was not included in surviving handoff, so1536-block reconciliation is historical-only.',historical_retained_summary=historical,original_879_gene_mapping_status='Original mapping hash/definition survives; mapping rows were not included in this handoff. No879-gene outcome analysis was originally completed.'),A3=dict(original_status='Unscored dependency closure',reconstructed_input_level_counts=support.to_dict('records'),outcome_test_performed=False),A5=dict(original_status='Unscored A2e-dependent dose test; distinct from completed A10',reconstructed_dose_levels=sorted(co.dose.unique().tolist()),outcome_test_performed=False),A6=dict(original_status='Unscored A2e-dependent family interaction',release_assignment_rows=len(inv),outcome_test_performed=False),A4=dict(original_status='Nonidentifiable modality contrast',design_columns=['intercept','Tahoe dataset','chemical modality'],design_rank=rank,number_columns=3,exact_modality_dataset_collinearity=True,outcome_test_performed=False))
    dump(out/'closure_diagnostics.json',status)
    sources=[root/'computed/tahoe/cohort.csv',root/'computed/tahoe/condition_controls.csv',root/'methods/sources.json',root/'originals/supporting/audit/data/prospective/tahoe_pseudobulk_parquet_manifest.json',root/'originals/supporting/audit/axis/frozen_axis_secon

,dataset,modality,model,representation,setting,context,source_path,source_sha256
0,Replogle,genetic CRISPRi,ST-SE-Replogle,SE,zeroshot,k562,ST-SE-Replogle/zeroshot/k562/eval_best.ckpt/k562_results.csv,eaf22cc6169fa98ab90913366529db232ac2109fe421599f66d47b14ffd183b3
1,Replogle,genetic CRISPRi,ST-SE-Replogle,SE,zeroshot,jurkat,ST-SE-Replogle/zeroshot/jurkat/eval_best.ckpt/jurkat_results.csv,372db67f297565e5542cac1bda688586c07e03e92a999bea953a9df812ba5317
2,Replogle,genetic CRISPRi,ST-SE-Replogle,SE,zeroshot,hepg2,ST-SE-Replogle/zeroshot/hepg2/eval_best.ckpt/hepg2_results.csv,9b7b329faa11cfdfbbac37ad571bdf47e8af7f988d61561eab51d80dce9d643a
3,Replogle,genetic CRISPRi,ST-SE-Replogle,SE,zeroshot,rpe1,ST-SE-Replogle/zeroshot/rpe1/eval_best.ckpt/rpe1_results.csv,ce7034b9e7af7255f44347ac9fdcbdd41e13b8ce674cab3c2700c0b41a1b1ffa
4,Replogle,genetic CRISPRi,ST-HVG-Replogle,HVG,zeroshot,k562,ST-HVG-Replogle/zeroshot/k562/eval_best.ckpt/k562_results.csv,8bd145c4667e03c9e367079dcfb38126e4e1f06b562f8a3fa6d1eaa1d0ba5821
5,Replogle,genetic CRISPRi,ST-HVG-Replogle,HVG,zeroshot,jurkat,ST-HVG-Replogle/zeroshot/jurkat/eval_best.ckpt/jurkat_results.csv,3efc817549bbf24edd6e427fb99466f58de8fadb3a4b4335523f5cfac1bc8098
6,Replogle,genetic CRISPRi,ST-HVG-Replogle,HVG,zeroshot,hepg2,ST-HVG-Replogle/zeroshot/hepg2/eval_best.ckpt/hepg2_results.csv,0c930ad038b56c6f43d25896a15cd68e121d8902e5457718b4dfc9f163f10604
7,Replogle,genetic CRISPRi,ST-HVG-Replogle,HVG,zeroshot,rpe1,ST-HVG-Replogle/zeroshot/rpe1/eval_best.ckpt/rpe1_results.csv,c187b3a026c8381a84acbe5035efa30947a6774c82a6519749b1b37e85fdd14d
8,Tahoe,chemical drug,ST-SE-Tahoe,SE,fewshot,C32,ST-SE-Tahoe/fewshot/state_generalization_X_state/eval_best.ckpt/C32_results.csv,9fee724f7d5dc46a3e0061e264347ce4c8c5c29b6c0dac2dc67303bcae25894a
9,Tahoe,chemical drug,ST-SE-Tahoe,SE,fewshot,HOP62,ST-SE-Tahoe/fewshot/state_generalization_X_state/eval_best.ckpt/HOP62_results.csv,646052bbdb9844e1a2b94f19c095d0f5a07db9529010f41252c61ba228544d56


**Calculated stage:** closures

<IPython.core.display.JSON object>

## 3. Axis

The proposed axis is genetic CRISPRi versus chemical perturbation. In these available releases it is exactly tied to dataset, with associated context/preprocessing/checkpoint differences. The source table shows those assignments directly.

In [3]:
display(pd.crosstab(assignments.dataset,assignments.modality))
display(assignments[['dataset','modality','model','representation','setting']].drop_duplicates())


modality,chemical drug,genetic CRISPRi
dataset,,
Replogle,0,8
Tahoe,15,0


,dataset,modality,model,representation,setting
0,Replogle,genetic CRISPRi,ST-SE-Replogle,SE,zeroshot
4,Replogle,genetic CRISPRi,ST-HVG-Replogle,HVG,zeroshot
8,Tahoe,chemical drug,ST-SE-Tahoe,SE,fewshot
13,Tahoe,chemical drug,ST-SE-Tahoe,SE,zeroshot
18,Tahoe,chemical drug,ST-HVG-Tahoe,HVG,zeroshot


## 4. Levels and eligibility

The two modality labels do not form independently varying experimental levels within a fixed dataset. The diagnostic matrix therefore contains intercept, dataset indicator and an identical modality indicator. All rows arise from the real release manifest.

In [4]:
dataset=(assignments.dataset=='Tahoe').astype(int)
modality=(assignments.modality=='chemical drug').astype(int)
design=pd.DataFrame({'intercept':1,'Tahoe_dataset':dataset,'chemical_modality':modality})
display(design)
assert np.array_equal(dataset,modality)


,intercept,Tahoe_dataset,chemical_modality
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0
5,1,0,0
6,1,0,0
7,1,0,0
8,1,1,1
9,1,1,1


## 5. Model / released scores

No original isolated modality outcome test was performed because the design cannot separate modality from dataset. The source inventory is the relevant evidence at this stage; no released score is repurposed as a causal modality estimate.

In [5]:
diagnostic=wf.read_json(ROOT,'computed/closures/closure_diagnostics.json')['A4']
display(JSON(diagnostic))
assert diagnostic['outcome_test_performed'] is False


<IPython.core.display.JSON object>

## 6. Curves, controls and diagnostics

Recompute matrix rank from the source-built assignment design. Rank below the number of columns is the actual diagnostic that prevents a separate modality coefficient. A performance curve would answer a different question and was not part of the original run.

In [6]:
calculated_rank=int(np.linalg.matrix_rank(design.to_numpy(float)))
assert calculated_rank==diagnostic['design_rank']
display(pd.DataFrame([{'design_columns':design.shape[1],'recomputed_rank':calculated_rank,'dataset_equals_modality':bool(np.array_equal(dataset,modality))}]))


,design_columns,recomputed_rank,dataset_equals_modality
0,3,2,True


## 7. Historical result and decision

The original candidate was closed as non-identifiable in the available releases. Modality could describe external transport of a later result, but no standalone within-model modality effect was claimed. The calculated source-design diagnostic above supports that recorded stopping decision.

In [7]:
wf.original_rows(ROOT,['A4'])
print('No modality performance curve or outcome coefficient was originally estimated; this notebook reproduces the source-design diagnostic instead.')


,candidate_id,axis,status,decision_reason,next_action
5,A4,perturbation modality,non_identifiable,modality is perfectly confounded with dataset context preprocessing and checkpoint across the av...,closed without a causal modality claim


These are attributed original decisions, not new adjudications.
No modality performance curve or outcome coefficient was originally estimated; this notebook reproduces the source-design diagnostic instead.
